# Max-Mix Purification Entropy Contours

Runs GPU Markov purification dynamics from the maximally mixed state for perfect-correction and post-selection protocols, then saves the sample-averaged whole-system spatial entropy contour for every cycle, including the initial state.


## Configurations And Domain-Wall Geometry

This Colab notebook generates the active maxmix purification campaign configured in the constants cell below. With the current constants, it writes the `N16x32_C100_dwtrunc0` campaign:

| Nx | Ny | nshell values | protocols / samples | cycles | init state | dtype | dw_truncation | alpha_1 / alpha_2 |
|---:|---:|---|---|---:|---|---|---|---|
| 16 | 32 | 1, 2 | perfect_correction / 10; postselect / 1 | 100 | maxmix | complex128 | False | 1 / 30 |

Domain-wall convention for this system size: `Nx=16` gives `half=Nx//2=8` and `w=floor(0.2*Nx)=3`, so the code stores `DW_loc=[5, 11]`. The topological `alpha_1` slab is `x=5..11` inclusive, the trivial `alpha_2` regions are `x=0..4` and `x=12..15`, and the two domain-wall locations are reported at `x=5` and `x=11` in the simulation metadata/convention. These walls extend along the full periodic `y` direction for `Ny=32`.

To generate the comparison campaign `N16x32_C100_dwtrunc1`, keep the same table values but set `DW_TRUNCATION = True` before running the notebook.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_small_system_testing'

def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'classA_U1FGTN_gpu.py').exists()

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with src/classA_U1FGTN_gpu.py')

SRC_DIR = BUNDLE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')

In [ ]:
import gc
import os
from typing import Any

import numpy as np
from tqdm.auto import tqdm
import torch

from classA_U1FGTN_gpu import classA_U1FGTN_gpu

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)

print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')

In [ ]:
NX = 16
NY = 32
NSHELL_VALUES = [1, 2]
SAMPLES = 10
CYCLES = 100
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = False
INIT_MODE = 'maxmix'
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
BATCH_SIZE = None
ENTROPY_SAMPLE_CHUNK = 1
EPS = 1e-12

PROTOCOL_CONFIGS = [
    {
        'protocol': 'perfect_correction',
        'postselect': False,
        'perfect_correction': True,
        'samples': SAMPLES,
    },
    {
        'protocol': 'postselect',
        'postselect': True,
        'perfect_correction': False,
        'samples': 1,
    },
]

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'purification_entropy_contours_maxmix'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
CAMPAIGN_ID = f"N{NX}x{NY}_C{CYCLES}_{'dwtrunc1' if DW_TRUNCATION else 'dwtrunc0'}"
CAMPAIGN_ROOT = OUTPUT_ROOT / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    {
        'Nx': NX,
        'Ny': NY,
        'nshell': int(nshell),
        **protocol_cfg,
    }
    for nshell in NSHELL_VALUES
    for protocol_cfg in PROTOCOL_CONFIGS
]

CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'

print(json.dumps({
    'campaign_id': CAMPAIGN_ID,
    'configs': CONFIGS,
    'cycles': CYCLES,
    'saved_cycles': list(range(CYCLES + 1)),
    'protocols': PROTOCOL_CONFIGS,
    'init_mode': INIT_MODE,
    'dw_truncation': DW_TRUNCATION,
    'dtype': DTYPE,
    'entropy_sample_chunk': ENTROPY_SAMPLE_CHUNK,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'output_root': str(OUTPUT_ROOT),
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'note': 'postselect=True uses one deterministic trajectory because the GPU driver forces samples=1.',
}, indent=2))


In [ ]:
def write_json_atomic(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    with tmp_path.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp_path.replace(path)


def save_npz_atomic(path: Path, **arrays: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    with tmp_path.open('wb') as fh:
        np.savez_compressed(fh, **arrays)
    tmp_path.replace(path)


def rel_to_gpu_data(path) -> str | None:
    if path is None:
        return None
    try:
        return str(Path(path).resolve().relative_to(GPU_DATA_ROOT.resolve()))
    except Exception:
        return None


def update_gpu_data_index() -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        with index_path.open('r', encoding='utf-8') as fh:
            payload = json.load(fh)
    else:
        payload = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}
    campaigns = payload.setdefault('campaigns', [])
    entry = {
        'name': CAMPAIGN_NAME,
        'kind': 'derived_entropy_observable',
        'manifest': f'{CAMPAIGN_NAME}/latest_campaign.json',
    }
    campaigns[:] = [item for item in campaigns if item.get('name') != CAMPAIGN_NAME]
    campaigns.append(entry)
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    write_json_atomic(index_path, payload)


def dw_truncation_label() -> str:
    return 'dwtrunc1' if DW_TRUNCATION else 'dwtrunc0'


def case_output_dir(cfg: dict) -> Path:
    return RUNS_ROOT / f"nsh{cfg['nshell']}_{cfg['protocol']}"


def entropy_contour_batch_torch(G_batch: torch.Tensor, *, nx: int, ny: int) -> torch.Tensor:
    if G_batch.ndim != 3:
        raise ValueError(f'Expected G_batch with shape (B,N,N), got {tuple(G_batch.shape)}')
    batch, nlayer, nlayer_2 = G_batch.shape
    expected = 2 * int(nx) * int(ny)
    if nlayer != nlayer_2 or nlayer != expected:
        raise ValueError(f'Expected covariance shape (B,{expected},{expected}), got {tuple(G_batch.shape)}')
    if not torch.isfinite(G_batch).all():
        bad = torch.nonzero(~torch.isfinite(G_batch), as_tuple=False)[0].detach().cpu().tolist()
        raise FloatingPointError(f'Non-finite covariance before entropy computation at index {bad}')

    occ = G_batch.clone()
    diag = occ.diagonal(dim1=-2, dim2=-1)
    diag.add_(1.0)
    occ.mul_(0.5)
    occ = 0.5 * (occ + occ.conj().transpose(-2, -1))

    evals, vecs = torch.linalg.eigh(occ)
    entropy_eps = max(EPS, 10.0 * float(torch.finfo(evals.real.dtype).eps))
    evals = torch.clamp(evals.real, entropy_eps, 1.0 - entropy_eps)
    weights = -(evals * torch.log(evals) + (1.0 - evals) * torch.log(1.0 - evals))
    diag_f = torch.sum(torch.abs(vecs) ** 2 * weights.unsqueeze(-2), dim=-1).real
    contour = diag_f.reshape(batch, ny, nx, 2).sum(dim=-1).transpose(1, 2).contiguous()
    if not torch.isfinite(contour).all():
        bad = torch.nonzero(~torch.isfinite(contour), as_tuple=False)[0].detach().cpu().tolist()
        raise FloatingPointError(f'Non-finite entropy contour at index {bad}')
    return contour


def new_contour_accumulator(*, cycles: int, nx: int, ny: int) -> dict[str, np.ndarray]:
    time_count = int(cycles) + 1
    return {
        'entropy_contour_sum': np.zeros((time_count, nx, ny), dtype=np.float64),
        'entropy_contour_sumsq': np.zeros((time_count, nx, ny), dtype=np.float64),
        'entropy_total_sum': np.zeros((time_count,), dtype=np.float64),
        'entropy_total_sumsq': np.zeros((time_count,), dtype=np.float64),
        'sample_counts_by_cycle': np.zeros((time_count,), dtype=np.int64),
        'cycles': np.arange(time_count, dtype=np.int64),
    }


def observe_entropy_contour(
    *,
    accumulator: dict[str, np.ndarray],
    cycle: int,
    G: torch.Tensor,
    nx: int,
    ny: int,
    sample_chunk: int,
) -> None:
    cycle = int(cycle)
    sample_chunk = max(1, int(sample_chunk))
    observed = 0
    for start in range(0, int(G.shape[0]), sample_chunk):
        stop = min(int(G.shape[0]), start + sample_chunk)
        contour = entropy_contour_batch_torch(G[start:stop], nx=nx, ny=ny).detach().cpu().numpy().astype(np.float64)
        total = contour.sum(axis=(1, 2))
        accumulator['entropy_contour_sum'][cycle] += contour.sum(axis=0)
        accumulator['entropy_contour_sumsq'][cycle] += np.square(contour).sum(axis=0)
        accumulator['entropy_total_sum'][cycle] += total.sum()
        accumulator['entropy_total_sumsq'][cycle] += np.square(total).sum()
        observed += contour.shape[0]
        del contour, total
        torch.cuda.empty_cache()
    accumulator['sample_counts_by_cycle'][cycle] += observed


def finalize_contour_accumulator(accumulator: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    counts = accumulator['sample_counts_by_cycle'].astype(np.float64)
    if np.any(counts <= 0):
        missing = np.flatnonzero(counts <= 0).tolist()
        raise RuntimeError(f'Missing entropy contour observations for cycles {missing}')
    counts_3d = counts[:, None, None]
    contour_mean = accumulator['entropy_contour_sum'] / counts_3d
    contour_var = np.zeros_like(contour_mean)
    multi = counts > 1
    contour_var[multi] = (
        accumulator['entropy_contour_sumsq'][multi]
        - np.square(accumulator['entropy_contour_sum'][multi]) / counts_3d[multi]
    ) / (counts[multi, None, None] - 1.0)
    contour_var = np.maximum(contour_var, 0.0)
    contour_std = np.sqrt(contour_var)
    contour_sem = contour_std / np.sqrt(counts_3d)

    total_mean = accumulator['entropy_total_sum'] / counts
    total_var = np.zeros_like(total_mean)
    total_var[multi] = (
        accumulator['entropy_total_sumsq'][multi]
        - np.square(accumulator['entropy_total_sum'][multi]) / counts[multi]
    ) / (counts[multi] - 1.0)
    total_var = np.maximum(total_var, 0.0)
    total_std = np.sqrt(total_var)
    total_sem = total_std / np.sqrt(counts)

    return {
        'entropy_contour_avg': contour_mean,
        'entropy_contour_std': contour_std,
        'entropy_contour_sem': contour_sem,
        'entropy_contour_sum': accumulator['entropy_contour_sum'],
        'entropy_contour_sumsq': accumulator['entropy_contour_sumsq'],
        'entropy_total_mean': total_mean,
        'entropy_total_std': total_std,
        'entropy_total_sem': total_sem,
        'entropy_total_sum': accumulator['entropy_total_sum'],
        'entropy_total_sumsq': accumulator['entropy_total_sumsq'],
        'sample_counts_by_cycle': accumulator['sample_counts_by_cycle'],
        'cycles': accumulator['cycles'],
    }


In [ ]:
def run_one_config(cfg: dict) -> dict:
    out_dir = case_output_dir(cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    run_samples = int(cfg['samples'])
    protocol = str(cfg['protocol'])
    postselect = bool(cfg['postselect'])
    perfect_correction = bool(cfg['perfect_correction'])
    print(
        f"[config] Nx={cfg['Nx']} Ny={cfg['Ny']} nshell={cfg['nshell']} "
        f"protocol={protocol} init={INIT_MODE} cycles={CYCLES} samples={run_samples} output={out_dir}"
    )

    model = classA_U1FGTN_gpu(
        Nx=int(cfg['Nx']),
        Ny=int(cfg['Ny']),
        DW=True,
        nshell=int(cfg['nshell']),
        filling_frac=0.5,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        trial_orbitals='X',
        dw_truncation=DW_TRUNCATION,
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )

    contour_accumulator = new_contour_accumulator(cycles=CYCLES, nx=int(cfg['Nx']), ny=int(cfg['Ny']))
    contour_pbar = tqdm(total=run_samples * (CYCLES + 1), desc='Entropy contours', unit='sample-cycle', leave=False)

    def contour_observer(*, cycle, G, batch_index, batch_start, batch_count):
        observe_entropy_contour(
            accumulator=contour_accumulator,
            cycle=cycle,
            G=G,
            nx=int(cfg['Nx']),
            ny=int(cfg['Ny']),
            sample_chunk=ENTROPY_SAMPLE_CHUNK,
        )
        contour_pbar.update(int(batch_count))

    try:
        result = model.run_markov_circuit(
            G_history=False,
            progress=False,
            cycles=CYCLES,
            postselect=postselect,
            perfect_correction=perfect_correction,
            samples=run_samples,
            init_mode=INIT_MODE,
            save=False,
            save_init=False,
            save_history_stride=None,
            n_a=0.5,
            sequence=SEQUENCE,
            batch_size=BATCH_SIZE,
            return_data=False,
            cycle_observer=contour_observer,
        )
    finally:
        contour_pbar.close()

    actual_samples = int(result['samples'])
    batch_size = int(result['batch_size'])
    batch_size_mode = result['batch_size_mode']
    batch_size_auto_info = result.get('batch_size_auto_info')

    contour_data = finalize_contour_accumulator(contour_accumulator)
    if not np.all(contour_data['sample_counts_by_cycle'] == actual_samples):
        raise RuntimeError(
            f"Expected {actual_samples} contour observations per cycle, got "
            f"{contour_data['sample_counts_by_cycle'].tolist()}"
        )
    del result, contour_accumulator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    metadata = {
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'campaign_id': CAMPAIGN_ID,
        'campaign_root': str(CAMPAIGN_ROOT),
        'campaign_root_relative': rel_to_gpu_data(CAMPAIGN_ROOT),
        'run_dir': str(out_dir),
        'run_dir_relative': rel_to_gpu_data(out_dir),
        'saved_observable': 'sample-averaged whole-system spatial entropy contour',
        'covariance_history_saved': False,
        'contour_collection_mode': 'cycle_observer_streaming',
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'nshell': int(cfg['nshell']),
        'samples_requested': run_samples,
        'samples': actual_samples,
        'cycles': CYCLES,
        'saved_cycles': contour_data['cycles'].astype(int).tolist(),
        'protocol': protocol,
        'postselect': postselect,
        'perfect_correction': perfect_correction,
        'init_mode': INIT_MODE,
        'alpha_1': ALPHA_1,
        'alpha_2': ALPHA_2,
        'dw_truncation': DW_TRUNCATION,
        'dtype_requested': DTYPE,
        'dtype_resolved': 'complex128',
        'device': DEVICE,
        'backend_requested': BACKEND,
        'backend_resolved': str(model.backend),
        'sequence': SEQUENCE,
        'batch_size_requested': BATCH_SIZE,
        'batch_size': batch_size,
        'batch_size_mode': batch_size_mode,
        'batch_size_auto_info': batch_size_auto_info,
        'entropy_sample_chunk': ENTROPY_SAMPLE_CHUNK,
    }

    npz_path = out_dir / 'entropy_contour_time_maxmix.npz'
    metadata['npz_path'] = str(npz_path)
    metadata['npz_path_relative'] = rel_to_gpu_data(npz_path)
    save_npz_atomic(
        npz_path,
        **contour_data,
        config_json=np.asarray(json.dumps(metadata, sort_keys=True)),
    )
    summary_path = out_dir / 'run_summary.json'
    summary = {
        **metadata,
        'summary_path': str(summary_path),
        'summary_path_relative': rel_to_gpu_data(summary_path),
    }
    write_json_atomic(summary_path, summary)
    print(f'[saved] {npz_path}')
    return summary


In [ ]:
all_results = []
for cfg in tqdm(CONFIGS, desc='Purification configs', unit='config'):
    all_results.append(run_one_config(cfg))

campaign_manifest = {
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'campaign_id': CAMPAIGN_ID,
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'output_root_relative': rel_to_gpu_data(OUTPUT_ROOT),
    'campaign_root_relative': rel_to_gpu_data(CAMPAIGN_ROOT),
    'runs_root_relative': rel_to_gpu_data(RUNS_ROOT),
    'configs': CONFIGS,
    'results': all_results,
    'samples_default': SAMPLES,
    'cycles': CYCLES,
    'saved_cycles': list(range(CYCLES + 1)),
    'protocols': PROTOCOL_CONFIGS,
    'init_mode': INIT_MODE,
    'dw_truncation': DW_TRUNCATION,
    'dtype': DTYPE,
    'saved_observable': 'sample-averaged whole-system spatial entropy contour',
    'covariance_history_saved': False,
    'contour_collection_mode': 'cycle_observer_streaming',
}
campaign_manifest_path = CAMPAIGN_ROOT / 'campaign_manifest.json'
write_json_atomic(campaign_manifest_path, campaign_manifest)
latest_campaign = {
    'name': CAMPAIGN_NAME,
    'kind': 'derived_entropy_observable',
    'active_campaign_id': CAMPAIGN_ID,
    'campaign_root': rel_to_gpu_data(CAMPAIGN_ROOT),
    'manifest': rel_to_gpu_data(campaign_manifest_path),
}
write_json_atomic(OUTPUT_ROOT / 'latest_campaign.json', latest_campaign)
update_gpu_data_index()
print(f'[done] campaign manifest: {campaign_manifest_path}')
print(f'[done] latest campaign: {OUTPUT_ROOT / "latest_campaign.json"}')


In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')